# Rover on Rough Terrain — Planning Challenge (30 min)

A point rover must drive from `start` to `goal` across rough 2D terrain.

**You have:**
- `field` `(H,W,3)` — terrain channels: **rock** (0/1, lethal), **mud** (0..1),
  **slope** (0..1). `x = column`, `y = row`.
- `demos` — a few expert `(x,y)` paths, each between its *own* start/goal (not
  yours). No single one is the answer; they reveal which terrain is costly.
- `astar_plan(cost_map)` — a **provided** planner. Give it a `(H,W)` cost map,
  get back a near-optimal path. You do **not** write a planner.

**Your one task:** write `cost_map_learned(field, demos)` → a `(H,W)` cost map
that, routed through `astar_plan`, yields a path as terrain-cheap as the experts'.
The provided `cost_map_baseline` ignores mud, so its path drives through the mud
and FAILs — that's the starting point you improve on.

### Setup &mdash; clone data &amp; import the prebuilt helpers

In [ ]:
import os, sys
import numpy as np

# Clone the data repo when running on Colab (no-op if files already exist). The
# prebuilt helpers and the data ship together in this repo.
if not os.path.exists('field_p1.npy'):
    import subprocess
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/vvanirudh/planning_challenge.git', '_data'],
        check=True)
    os.chdir('_data')

sys.path.insert(0, os.getcwd())
from challenge_utils_p1 import (field, demos, start, goal, H, W,
                                ROCK, MUD, SLOPE, ROCK_PENALTY,
                                show, path_cost, evaluate,
                                astar_plan, cost_map_baseline, api_help)
print('field', field.shape, '| demos', len(demos),
      '| start', start, 'goal', goal)

### API (all prebuilt — do not edit)

Everything below is imported for you; you only write `cost_map_learned`. The true
terrain cost is hidden behind the scorer — recovering it from the demos is the point.
`evaluate` PASSes if the path reaches the goal, avoids rocks, and its terrain cost is
within tolerance of optimal. Run the cell for the full list (`help(fn)` for details).

In [ ]:
api_help()

### Terrain + expert demos (gray)

In [ ]:
show(title='terrain (true cost) + expert demos (gray)')

### Baseline run &mdash; expect `FAIL`

In [ ]:
# Provided planner + wrong baseline cost: reaches the goal but FAILs the bar
# (the cost ignores mud). The planner is fine; the COST is what you'll fix.
cm_base = cost_map_baseline(field)
path = astar_plan(cm_base)
show(path, cost_map=cm_base, title='baseline cost: planned path')
evaluate(path, cm_base)

## TODO &mdash; `cost_map_learned` (see the cell)

In [ ]:
# ---- TODO: learn a (H,W) cost map from the demos -------------------------
# Make cells/terrain the experts drive through cheap, and what they detour
# around costly, so astar_plan routes like them. You only need the RANKING
# right, not the true cost numerically. Keep rocks at ROCK_PENALTY.
# (e.g. count demo visitation, or weight up mud/slope — your call.)
def cost_map_learned(field, demos):
    '''Return an (H,W) cost map inferred from demos. Keep rocks impassable.'''
    # TODO: use the demos. (Falls back to the wrong baseline for now.)
    return cost_map_baseline(field)

### Run with your learned cost &mdash; goal: `PASS`

In [ ]:
cm = cost_map_learned(field, demos)
path = astar_plan(cm)
show(path, cost_map=cm, title='learned: planned path over learned cost map')
show(path, title='learned: planned path over terrain')
evaluate(path, cm)   # goal: PASS

### Discussion (optional — no code needed)

Be ready to talk through: where your learned cost generalizes badly; A* vs.
sampling vs. trajectory-opt and what each guarantees; the planner's Euclidean
heuristic (admissible here?); 8- vs 4-connectivity and resolution tradeoffs;
adding a safety margin around rocks; replanning in a 50 Hz loop.